# 04 向量化与索引构建（全量）

> **目标**：对全量 `oa_comm_chunks.jsonl`（约 610 万 chunks）构建 BGE + ChromaDB 持久化索引。

> **本 notebook 为全量版**。输入与向量库均在外接盘；验证期请用 `vectorize-index.ipynb`（工程内 `data/chroma_db/`）。

## 存储策略（与第三阶段一致）

| 模式 | Notebook | 输入 | ChromaDB 位置 |
|------|----------|------|---------------|
| **验证** | `vectorize-index.ipynb` | `data/processed/chunks_sample.jsonl` | **工程内** `data/chroma_db/` |
| **全量** | 本文件 | `E:\...\oa_comm_chunks.jsonl` | **外接盘** `E:\...\chroma_db\` |

## 执行前检查

- 外接盘全量 chunks 已就绪
- **强烈建议**安装 CUDA 版 PyTorch
- 启用 `RESUME=True` 支持断点续传


---
## 【C0】环境配置 + 设备检测

检测并记录运行设备（GPU/CPU），作为日后对齐的背景信息。

In [ ]:
import sys
import os
import json
from pathlib import Path

SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if os.name == "nt":
    DATA_ROOT = Path("E:/med-llm-rag-datasets")
else:
    DATA_ROOT = Path("/Volumes/Lexar/med-llm-rag-datasets")

INPUT_JSONL = DATA_ROOT / "processed" / "oa_comm_chunks.jsonl"
PERSIST_DIR = DATA_ROOT / "chroma_db"
COLLECTION = "pmc_oa_comm_full"

print(f"数据根目录: {DATA_ROOT}")
print(f"输入文件: {INPUT_JSONL}")
print(f"向量库目录: {PERSIST_DIR} (外接盘，全量)")
print(f"collection: {COLLECTION}")
assert DATA_ROOT.exists(), f"数据根目录不存在: {DATA_ROOT}"
assert INPUT_JSONL.exists(), f"全量 chunks 不存在: {INPUT_JSONL}"


In [ ]:
# 设备检测（背景信息记录）
import torch

ENV_INFO = {
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}
if torch.cuda.is_available():
    ENV_INFO["gpu_name"] = torch.cuda.get_device_name(0)
    ENV_INFO["gpu_mem_GB"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)

print("=== 运行环境 ===")
for k, v in ENV_INFO.items():
    print(f"  {k}: {v}")

if not ENV_INFO["cuda_available"]:
    print("\n⚠️ 当前为 CPU 版 PyTorch，无法用 GPU。")
    print("   全量 610 万 chunks **必须**使用 GPU + CUDA 版 PyTorch，否则耗时极长。")

---
## 【C1】加载嵌入模型

加载 `bge-small-en-v1.5`，确认输出维度 = 384。首次运行会联网下载模型权重（约 130MB）。

In [ ]:
from embedder import DocumentEmbedder
import torch

BATCH_ENCODE = 256 if torch.cuda.is_available() else 128
embedder = DocumentEmbedder(model_name="BAAI/bge-small-en-v1.5", batch_size=BATCH_ENCODE)
print(f"encode batch_size: {BATCH_ENCODE}")
for k, v in embedder.device_info().items():
    print(f"  {k}: {v}")
print(f"\n嵌入维度: {embedder.dimension}")
assert embedder.dimension == 384


In [ ]:
# 快速验证：文档端 vs 查询端编码
doc_vec = embedder.encode_documents(["Diabetes is a chronic metabolic disease."])
qry_vec = embedder.encode_queries(["What is diabetes?"])
print(f"文档向量维度: {len(doc_vec[0])}")
print(f"查询向量维度: {len(qry_vec[0])}")
print(f"查询端已自动加指令前缀: '{embedder.query_instruction}'")

---
## 【C2】构建 ChromaDB 索引

分批读取 chunks，文档端编码后写入 ChromaDB（余弦相似度），支持断点续传。

| 参数 | 说明 |
|------|------|
| `BATCH_SIZE` | 每批编码+入库的 chunk 数 |
| `RESUME` | 断点续传（从 progress.json 续跑） |
| `RESET` | 设 True 会**清空**重建该 collection |

In [ ]:
BATCH_SIZE = 512\nRESUME = True\nRESET = False\n

In [ ]:
from index_builder import ChromaIndexBuilder

builder = ChromaIndexBuilder(
    persist_dir=PERSIST_DIR,
    collection_name=COLLECTION,
    embedder=embedder,
)

if RESET:
    builder.client.delete_collection(COLLECTION)
    builder = ChromaIndexBuilder(PERSIST_DIR, COLLECTION, embedder)
    # 同时清除进度文件
    pf = builder._progress_path()
    if pf.exists():
        pf.unlink()
    print("已重置 collection")

print(f"入库前 collection 计数: {builder.collection.count():,}")

In [ ]:
result = builder.build_from_jsonl(
    jsonl_path=INPUT_JSONL,
    batch_size=BATCH_SIZE,
    resume=RESUME,
)
print(f"\n入库完成，collection 共 {result['total_in_collection']:,} 条")

---\n## 【C3】保存索引统计\n\n保存到 `outputs/tables/`（全量）。\n

In [ ]:
import pandas as pd

sample_rows = []
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5000:
            break
        sample_rows.append(json.loads(line))
df_sample = pd.DataFrame(sample_rows)
token_stats = {
    "mean": round(float(df_sample["token_count"].mean()), 2),
    "max": int(df_sample["token_count"].max()),
    "min": int(df_sample["token_count"].min()),
    "note": "基于前 5000 chunks 抽样估计",
}
stats = builder.get_stats(chunk_token_stats=token_stats)
stats["data_type"] = "全量（oa_comm_chunks.jsonl）"
stats["input_file"] = str(INPUT_JSONL)
stats["env_info"] = ENV_INFO
out_path = Path("../outputs/tables/index_stats.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)
print(f"统计已保存: {out_path.resolve()}")


---
## 【C4】相似性检索验证

1. **自相似性**：从索引取一段文本作查询，应命中自身（距离最小）
2. **语义检索**：用自然语言医学问题检索相关片段

In [ ]:
sample_rec = df_sample.iloc[0]
self_query = sample_rec["text"][:300]
res = builder.query(self_query, n_results=3)
hit = res["ids"][0][0] == sample_rec["chunk_id"]
print(f"自相似性命中自身 ({sample_rec['chunk_id']}): {hit}")


In [ ]:
# 语义检索测试：自然语言医学问题
queries = [
    "What is the role of gene regulation in malaria parasites?",
    "conservation of Asian elephants",
    "circadian rhythm in Drosophila",
]
for q in queries:
    res = builder.query(q, n_results=2)
    print(f"\n查询: {q}")
    for cid, dist, title in zip(
        res["ids"][0], res["distances"][0],
        [m.get("source_title", "") for m in res["metadatas"][0]],
    ):
        print(f"  - [{dist:.3f}] {cid}  {title[:70]}")

---
## 【C5】边界情况 + 元数据过滤验证

In [ ]:
# 边界：空查询 / 超长查询
validation = {}

try:
    r_empty = builder.query("", n_results=3)
    validation["空查询"] = f"返回 {len(r_empty['ids'][0])} 条（未报错）"
except Exception as e:
    validation["空查询"] = f"异常: {type(e).__name__}"

try:
    r_long = builder.query("diabetes " * 2000, n_results=3)
    validation["超长查询"] = f"返回 {len(r_long['ids'][0])} 条（截断处理，未报错）"
except Exception as e:
    validation["超长查询"] = f"异常: {type(e).__name__}"

for k, v in validation.items():
    print(f"  {k}: {v}")

In [ ]:
# 元数据过滤：只在多块文献（strategy=sliding_window）中检索
res_filter = builder.query(
    "circadian rhythm",
    n_results=3,
    where_filter={"strategy": "sliding_window"},
)
print("过滤条件: strategy = sliding_window")
print(f"返回 {len(res_filter['ids'][0])} 条:")
for cid, meta in zip(res_filter["ids"][0], res_filter["metadatas"][0]):
    print(f"  - {cid}  strategy={meta.get('strategy')}  total_chunks={meta.get('total_chunks')}")

all_sw = all(m.get("strategy") == "sliding_window" for m in res_filter["metadatas"][0])
print(f"\n元数据过滤生效: {'✅ 是' if all_sw else '⚠️ 否'}")

In [ ]:
validation_report = {
    "验证日期": pd.Timestamp.now().isoformat(),
    "数据类型": "全量",
    "collection": COLLECTION,
    "向量数量": builder.collection.count(),
    "自相似性命中自身": bool(hit),
    "边界情况": validation,
    "元数据过滤生效": bool(all_sw),
}
rep_path = Path("../outputs/tables/query_validation.json")
with open(rep_path, "w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)
print(f"验证报告: {rep_path.resolve()}")


---\n## 完成（全量）\n\n| 产物 | 路径 | Git |\n|------|------|-----|\n| 向量库 | `E:\\med-llm-rag-datasets\\chroma_db\\` | ❌ 外接盘 |\n| 索引统计 | `outputs/tables/index_stats.json` | ✅ |\n| 查询验证 | `outputs/tables/query_validation.json` | ✅ |\n